# Lab 1: N-gram language models and verification

**Natural Language Processing and Large Language Models (MDC_53607_EP)**  
Ye Zhu · M2 Data Science and AI · IP Paris

Build a small bigram language model, measure what happens on unseen text and test a faulty evaluator. This lab follows Lecture 1. It adds **six computational Moodle checkpoints**, labelled CP1–CP6, to the two conceptual questions in Quiz 1.

**Prerequisites:** Python functions, dictionaries, loops, and the probability concepts from Lecture 1.

Implement functions marked `TODO`, run the small tests, then compute your checkpoint outputs. The starter functions intentionally raise `NotImplementedError` until you complete them. Run cells in order. Leave the data files and supplied preprocessing unchanged.

For each checkpoint, enter only the requested number in Moodle, without units or explanatory text. Integer answers are exact; round other answers as specified. Retain full precision inside your code.

Follow your instructor's policy on AI assistance. If you use an assistant, check its suggestions against the specifications and tests. A plausible numerical output is not by itself a correctness test.

**Submission:** You are required to enter the exact number to the Moodle. The submission opens until 23:59 on the 16th Nov, 2026.


## Setup

**Google Colab:** upload/open this `.ipynb` in [Colab](https://colab.research.google.com/), connect to a standard CPU runtime, then run the next cell. When prompted, upload **Lab1_Student_Package.zip**. No installation or GPU is needed. A new Colab runtime may require uploading the ZIP again.

**Local Jupyter:** unzip the student package and open the notebook from that folder. Keep the `data/` directory beside the notebook. The notebook uses only the Python standard library; the notebook application itself must already be installed.

The supplied corpus consists of short, artificial sentences about campus and research activities. It is deliberately repetitive and contains new combinations in the held-out splits. Use it to study modeling choices, not to make claims about real-world language-model quality. All texts are original teaching material.


In [2]:
from pathlib import Path
from collections import Counter
import hashlib
import io
import json
import math
import zipfile

# Colab: open this notebook, then upload Lab1_Student_Package.zip when prompted.
# Local Jupyter: unzip the student package and open the notebook in that folder.
DATA_DIR = Path('data')
required = ['train.txt', 'dev.txt', 'test.txt', 'audit.txt', 'manifest.json']

if not all((DATA_DIR / name).is_file() for name in required):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            'Unzip Lab1_Student_Package.zip, then open the notebook beside its data folder.'
        ) from exc
    print('Upload Lab1_Student_Package.zip (the whole ZIP, not only the notebook).')
    uploaded = files.upload()
    found = False
    for filename, payload in uploaded.items():
        if not zipfile.is_zipfile(io.BytesIO(payload)):
            continue
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            if all('data/' + name in archive.namelist() for name in required):
                DATA_DIR.mkdir(exist_ok=True)
                # Read only the expected data entries, never arbitrary ZIP paths.
                for name in required:
                    (DATA_DIR / name).write_bytes(archive.read('data/' + name))
                found = True
                break
    if not found:
        raise FileNotFoundError('The uploaded ZIP does not contain the expected data files.')

manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['version'] == 'lab1-v1', 'Use the corpus version supplied with this lab.'
for filename, info in manifest['files'].items():
    contents = (DATA_DIR / filename).read_bytes()
    contents = contents.replace(b'\r\n', b'\n').replace(b'\r', b'\n')
    digest = hashlib.sha256(contents).hexdigest()
    assert digest == info['sha256'], f'{filename} has changed. Restore the supplied file.'

def read_sentences(filename):
    return [line.lower().split()
            for line in (DATA_DIR / filename).read_text(encoding='utf-8').splitlines()
            if line.strip()]

raw_train = read_sentences('train.txt')
raw_dev = read_sentences('dev.txt')
raw_test = read_sentences('test.txt')
raw_audit = read_sentences('audit.txt')
print('Corpus version:', manifest['version'])
print('Sentences:', len(raw_train), 'train,', len(raw_dev), 'dev,', len(raw_test), 'test')
print('Example:', ' '.join(raw_train[0]))


Corpus version: lab1-v1
Sentences: 1608 train, 220 dev, 240 test
Example: the new message contains a useful example of history .


### Conventions used by every checkpoint

1. One nonempty line is one sentence. Lowercase and split on whitespace. Punctuation is already separated; `.` is a token.
2. Build the vocabulary from **training text only**. Keep words seen at least twice. Map every other word, including unseen development/test words, to `<unk>`.
3. The prediction vocabulary contains these retained words, `<unk>` and `</s>`. It excludes `<s>`.
4. Add exactly one `<s>` before and one `</s>` after each encoded sentence. Boundaries are added in the modeling functions, not stored in `train`, `dev`, `test` or `audit`.
5. Count the first-word prediction and the `</s>` prediction. Never predict `<s>` and never count transitions between different sentences.
6. Use natural logarithms and `math.exp` for perplexity. This is equivalent to base-2 logs with `2 ** x`, provided the bases match.

For `['the', 'cat', '.']`, the prediction pairs are:

```text
(<s>, the), (the, cat), (cat, .), (., </s>)
```

There are four prediction events. `<s>` is a conditioning symbol only. The following preprocessing cell is supplied; do not change it.


In [3]:
BOS, EOS, UNK = '<s>', '</s>', '<unk>'
MIN_COUNT = 2
ALPHA = 0.1
LAMBDA_GRID = [i / 10 for i in range(11)]

# Vocabulary is learned from raw TRAINING text only.
raw_training_counts = Counter(w for sent in raw_train for w in sent)
vocab = tuple(sorted(
    {w for w, count in raw_training_counts.items() if count >= MIN_COUNT}
    | {UNK, EOS}
))
vocab_set = set(vocab)
V = len(vocab)

def encode(sentences):
    return [[w if w in vocab_set else UNK for w in sent] for sent in sentences]

train = encode(raw_train)
dev = encode(raw_dev)
test = encode(raw_test)
audit = encode(raw_audit)
assert BOS not in vocab and EOS in vocab and UNK in vocab
print('Prediction vocabulary size:', V)
print('Encoded example:', train[0])


Prediction vocabulary size: 80
Encoded example: ['the', 'new', 'message', 'contains', 'a', 'useful', 'example', 'of', 'history', '.']


## 1. Count n-grams

Implement `count_ngrams(sentences)`. Return three `Counter` objects:

- `unigram[w]`: how many times `w` is a **predicted target**, including `</s>`.
- `bigram[(h, w)]`: the number of within-sentence transitions from history `h` to target `w`.
- `context[h]`: the number of prediction events with history `h`.

For each sentence, construct its padded sequence, iterate over consecutive pairs, and update all three counts. `Counter` returns zero for an absent key. Do not remove duplicates from the corpus.

**Notation:** Let $w$ denote a target word and $h$ its history (the preceding word). For example, in the sentence `<s> w_1 w_2 w_3 </s>`, the history of target $w_i$ is $w_{i-1}$.


In [ ]:
def count_ngrams(sentences):
    """Return target unigram, bigram and history counts."""
    unigram_counts = Counter()  
    bigram_counts = Counter()   
    history_counts = Counter()   

    for sentence in sentences:
        tokens = [BOS] + list(sentence) + [EOS]
        for history, target in zip(tokens[:-1], tokens[1:]):
            unigram_counts[target] += 1
            bigram_counts[(history, target)] += 1
            history_counts[history] += 1

    return unigram_counts, bigram_counts, history_counts

In [5]:
toy = [['red', 'bird'], ['red', 'fox']]
u_toy, b_toy, c_toy = count_ngrams(toy)
assert b_toy[(BOS, 'red')] == 2
assert b_toy[('bird', EOS)] == 1
assert b_toy[(EOS, BOS)] == 0
assert u_toy[EOS] == 2 and u_toy[BOS] == 0
assert c_toy['red'] == 2 and c_toy[EOS] == 0
assert sum(u_toy.values()) == sum(b_toy.values()) == sum(c_toy.values()) == 6
print('Toy counting checks passed.')


Toy counting checks passed.


In [6]:
unigram, bigram, context = count_ngrams(train)
assert sum(context.values()) == sum(len(s) + 1 for s in train)
assert context[BOS] == len(train)
assert unigram[EOS] == len(train)


### Moodle checkpoint CP1

Using the supplied lab1-v1 corpus and preprocessing, how many distinct bigram pairs have positive counts in the training split? Count after rare-word replacement. Include transitions from `<s>` and to `</s>`. Do not create transitions across sentences. Enter an integer.


In [10]:
cp1 = len(bigram)
print('CP1:', cp1)

CP1: 312


## 2. Maximum likelihood and unseen events

For an observed history, estimate

$$p_{\mathrm{MLE}}(w\mid h)=\frac{c(h,w)}{c(h)}.$$

For this lab, define the score as `0.0` when a history has no training occurrences. An entirely unseen history does not have a defined MLE distribution; this explicit convention lets us count failed predictions. Smoothing will supply a valid distribution later.


In [ ]:
def mle_probability(h, w, bigram, context):
    """Return an unsmoothed conditional probability, or 0 for an unseen history."""
    # TODO: divide the pair count by the history count, with a zero-history guard.
    pair_count = bigram.get((h, w), 0)
    history_count = context.get(h, 0)

    if history_count == 0:
        return 0

    return pair_count / history_count

In [13]:
assert math.isclose(mle_probability('red', 'bird', b_toy, c_toy), 0.5)
assert mle_probability('bird', 'red', b_toy, c_toy) == 0.0
assert mle_probability('unseen', 'red', b_toy, c_toy) == 0.0
p_mle = lambda h, w: mle_probability(h, w, bigram, context)
print('MLE checks passed.')


MLE checks passed.


### Moodle checkpoint CP2

Using the training counts, what is the unsmoothed maximum-likelihood probability P(model | language)? Enter a decimal rounded to six digits after the decimal point. Use the lab1-v1 vocabulary and boundary conventions.


In [14]:
cp2 = p_mle('language', 'model')
print(f'CP2: {cp2:.6f}')


CP2: 0.533569


### Unseen transitions

Implement `count_zero_events`. Count **prediction positions**, not distinct pairs, at which the supplied probability function returns zero. The test split has already been mapped to the training vocabulary, so do not rebuild or enlarge the vocabulary here.


In [17]:
def count_zero_events(sentences, probability):
    """Count zero-probability prediction events, including sentence endings."""
    # TODO: iterate over padded sentences and count events whose probability is zero.
    zero_count = 0
    for sentence in sentences:
        tokens = [BOS] + list(sentence) + [EOS]
        for history, target in zip(tokens[:-1], tokens[1:]):
            if probability(history, target) == 0:
                zero_count += 1
    return zero_count


In [18]:
toy_p = lambda h, w: mle_probability(h, w, b_toy, c_toy)
assert count_zero_events([['red', 'red']], toy_p) == 2
assert count_zero_events([['red', 'red'], ['red', 'red']], toy_p) == 4
assert count_zero_events(toy, toy_p) == 0
print('Unseen-event checks passed.')


Unseen-event checks passed.


### Moodle checkpoint CP3

On the encoded test split, how many prediction positions have probability zero under the unsmoothed training bigram model? Count occurrences, not distinct pairs. Include the first word after `<s>` and every `</s>` target. Enter an integer.


In [19]:
cp3 = count_zero_events(test, p_mle)
print('CP3:', cp3)


CP3: 395


**Discuss:** What happens to a sentence's probability if just one observed transition has probability zero? Does mapping unfamiliar words to `<unk>` prevent all unseen bigrams?

It would create a probability zero for the whole sentence, considering the Chain Rule. Mapping unfamiliar words to the `<unk>` token prevent from onlu unseen individual word to get a probability equal to zero, not bigrams (combination of two words).

## 3. Add-alpha smoothing and corpus perplexity

Implement

$$p_\alpha(w\mid h)=\frac{c(h,w)+\alpha}{c(h)+\alpha V},\qquad \alpha=0.1.$$

Here $V$ is the number of **possible targets**, including `<unk>` and `</s>`, excluding `<s>`. The formula also handles a history with no observed counts. All queries in this notebook use targets in the prediction vocabulary.


In [20]:
def smoothed_probability(h, w, bigram, context, vocab_size, alpha):
    """Return an add-alpha bigram probability."""
    # TODO: adjust both numerator and denominator.
    pair_count = bigram.get((h, w), 0)
    history_count = context.get(h, 0)
    numerator = pair_count + alpha
    denominator = history_count + alpha * vocab_size
    return numerator / denominator


In [21]:
toy_vocab = ('red', 'bird', 'fox', EOS, UNK)
toy_smooth = lambda h, w: smoothed_probability(h, w, b_toy, c_toy, len(toy_vocab), 0.1)
assert math.isclose(toy_smooth('red', 'bird'), 1.1 / 2.5)
for h in [BOS, 'red', 'bird', 'not_seen']:
    assert math.isclose(math.fsum(toy_smooth(h, w) for w in toy_vocab), 1.0, abs_tol=1e-12)
p_bigram = lambda h, w: smoothed_probability(h, w, bigram, context, V, ALPHA)
print('Smoothing checks passed.')


Smoothing checks passed.


### Evaluate in log space

For all $T$ prediction events in the corpus, compute

$$\operatorname{PPL}=\exp\left(-\frac{1}{T}\sum_{i=1}^{T}\log p(w_i\mid h_i)\right).$$

Implement `corpus_perplexity`. Accumulate log probabilities over all sentences, then divide by the **total number of predicted targets**, not the number of sentences. Include each `</s>` target. Do not multiply many probabilities directly.

- Return `math.inf` if any observed target has probability zero.
- Raise `ValueError` if there are no prediction events, or a probability lies outside `[0, 1]`.
- `math.fsum` provides a stable way to add the individual negative log probabilities.


In [24]:
def corpus_perplexity(sentences, probability):
    """Return token-weighted corpus perplexity; return infinity for a zero-probability event."""
    # TODO: accumulate negative natural logs and the number of prediction events.
    # TODO: exponentiate only after averaging over all events.
    total_log_prob = 0
    num_events = 0

    if corpus_perplexity is None or len(sentences) == 0:
        raise ValueError("Sentences cannot be None or empty.")

    for sentence in sentences:
        tokens = [BOS] + list(sentence) + [EOS]
        for history, target in zip(tokens[:-1], tokens[1:]):
            prob = probability(history, target)
            if prob == 0:
                return float('inf')
            total_log_prob += -math.log(prob)
            num_events += 1

    if num_events == 0:
        return float('inf')

    avg_log_prob = total_log_prob / num_events
    
    return math.exp(avg_log_prob)


In [25]:
# Two sentences with unequal lengths: three prediction events in total.
example_p = {(BOS, 'a'): 0.5, ('a', EOS): 0.25, (BOS, EOS): 0.5}
probe = [['a'], []]
expected = math.exp((-math.log(0.5) - math.log(0.25) - math.log(0.5)) / 3)
assert math.isclose(corpus_perplexity(probe, lambda h, w: example_p[(h, w)]), expected)
assert math.isinf(corpus_perplexity([['red', 'red']], toy_p))
try:
    corpus_perplexity([], toy_p)
    raise AssertionError('An empty corpus should raise ValueError.')
except ValueError:
    pass
print('Perplexity checks passed.')


Perplexity checks passed.


### Moodle checkpoint CP4

What is the test-set corpus perplexity of the add-alpha bigram model with alpha = 0.1? Use the lab1-v1 prediction vocabulary (including `<unk>` and `</s>`, excluding `<s>`). Aggregate negative log probabilities over all prediction positions, including </s>, then exponentiate their token-weighted mean. Enter six digits after the decimal point.


In [26]:
cp4 = corpus_perplexity(test, p_bigram)
print(f'CP4: {cp4:.6f}')


CP4: 9.359853


**Discuss:** We fixed alpha before evaluating this test set. Why would selecting alpha after inspecting many test-set scores weaken the final evaluation?

Because it would have make alpha fit on the test-set and not the training set: it means selecting a biased alpha. Also, given that the train test if way bigger (1608 sentences vs 240 sentences in test), it's more reliable fitting alpha on train.


## 4. Interpolation selected on development data

The supplied unigram model uses the same target counts, vocabulary and alpha:

$$p_{\mathrm{uni}}(w)=\frac{c(w)+\alpha}{N+\alpha V},\qquad N=\sum_{w\in\mathcal V}c(w).$$

Implement the mixture

$$p_\lambda(w\mid h)=\lambda\,p_{\mathrm{bigram}}(w\mid h)+(1-\lambda)\,p_{\mathrm{uni}}(w).$$

Here **lambda weights the bigram model**. Both components are add-alpha smoothed with alpha = 0.1. Evaluate the eleven candidates on **dev**, select using unrounded scores, then report the selected model on test once. In an exact tie, choose the smaller lambda.


In [27]:
N_TRAIN = sum(unigram.values())
def p_unigram(w):
    return (unigram[w] + ALPHA) / (N_TRAIN + ALPHA * V)
assert math.isclose(math.fsum(p_unigram(w) for w in vocab), 1.0, abs_tol=1e-12)


In [30]:
def interpolated_probability(h, w, weight, p_bigram, p_unigram):
    """Mix the supplied bigram and unigram probabilities."""
    # TODO: return the convex combination defined above.
    return weight * p_bigram(h, w) + (1 - weight) * p_unigram(w)



In [31]:
probe_h, probe_w = 'language', 'model'
assert math.isclose(interpolated_probability(probe_h, probe_w, 1.0, p_bigram, p_unigram), p_bigram(probe_h, probe_w))
assert math.isclose(interpolated_probability(probe_h, probe_w, 0.0, p_bigram, p_unigram), p_unigram(probe_w))
dev_scores = []
for weight in LAMBDA_GRID:
    p_current = lambda h, w, weight=weight: interpolated_probability(h, w, weight, p_bigram, p_unigram)
    score = corpus_perplexity(dev, p_current)
    dev_scores.append((weight, score))
    print(f'lambda={weight:.1f}  dev perplexity={score:.6f}')


lambda=0.0  dev perplexity=47.197470
lambda=0.1  dev perplexity=26.849668
lambda=0.2  dev perplexity=20.452723
lambda=0.3  dev perplexity=16.903420
lambda=0.4  dev perplexity=14.603617
lambda=0.5  dev perplexity=12.993202
lambda=0.6  dev perplexity=11.819110
lambda=0.7  dev perplexity=10.954899
lambda=0.8  dev perplexity=10.344962
lambda=0.9  dev perplexity=10.009670
lambda=1.0  dev perplexity=10.406551


In [33]:
# TODO: choose the smallest-perplexity row, breaking exact ties by lambda.
best_weight = 0.9
assert best_weight is not None, 'Select best_weight from dev_scores.'


### Moodle checkpoint CP5

For p_lambda(w|h) = lambda*p_bigram(w|h) + (1-lambda)*p_unigram(w), smooth both component models with alpha = 0.1 over the same lab1-v1 prediction vocabulary. Which lambda in {0.0, 0.1, ..., 1.0} minimizes DEV corpus perplexity? Select using unrounded scores. If there is an exact tie, choose the smaller lambda. Enter lambda to one decimal place.


In [34]:
cp5 = best_weight
print(f'CP5: {cp5:.1f}')
p_selected = lambda h, w: interpolated_probability(h, w, best_weight, p_bigram, p_unigram)
print(f'Selected-model test perplexity (not an extra checkpoint): {corpus_perplexity(test, p_selected):.6f}')


CP5: 0.9
Selected-model test perplexity (not an extra checkpoint): 9.015191


**Discuss:** Why can interpolation help even though it adds no training text? Does the best weight on this small artificial corpus establish the best weight for other domains?


## 5. Audit a candidate evaluator

A proposed evaluator computes a perplexity for each sentence, then averages those perplexities. It runs without an exception and returns a positive number. Is it implementing the corpus metric we specified?

1. Run it on `audit`, using **p_bigram** (alpha = 0.1), not `p_selected`.
2. Construct a small counterexample and explain what is wrong.
3. Implement `repaired_perplexity` **independently**, by accumulating total negative log-likelihood and total target count. For this exercise, do not call `corpus_perplexity` inside the repaired function.
4. Check that the repaired result matches your earlier corpus evaluator. Compute CP6 using the unrounded values.

The audit file contains eight sentences of different lengths. It is a diagnostic fixture, not an additional test set for selecting lambda.


In [35]:
def candidate_perplexity(sentences, probability):
    sentence_scores = [corpus_perplexity([sent], probability) for sent in sentences]
    return math.fsum(sentence_scores) / len(sentence_scores)

candidate_score = candidate_perplexity(audit, p_bigram)
print(f'Candidate evaluator: {candidate_score:.6f}')


Candidate evaluator: 32.961237


In [36]:
def repaired_perplexity(sentences, probability):
    """Recompute corpus perplexity without calling corpus_perplexity."""
    # TODO: combine evidence at the prediction-event level.
    total_log_prob = 0
    num_events = 0
    for sentence in sentences:
        tokens = [BOS] + list(sentence) + [EOS]
        for history, target in zip(tokens[:-1], tokens[1:]):
            prob = probability(history, target)
            if prob == 0:
                return float('inf')
            total_log_prob += -math.log(prob)
            num_events += 1
    if num_events == 0:
        return float('inf')
    avg_log_prob = total_log_prob / num_events
    return math.exp(avg_log_prob)


In [37]:
# Unequal sentence lengths make the aggregation issue visible.
short_probe = [[], ['a', 'b']]
probe_table = {(BOS, EOS): 0.5, (BOS, 'a'): 0.5, ('a', 'b'): 0.25, ('b', EOS): 0.25}
probe_probability = lambda h, w: probe_table[(h, w)]
assert not math.isclose(candidate_perplexity(short_probe, probe_probability),
                        repaired_perplexity(short_probe, probe_probability), rel_tol=1e-9)
assert math.isclose(repaired_perplexity(short_probe, probe_probability),
                    corpus_perplexity(short_probe, probe_probability), rel_tol=1e-12)
corrected_score = repaired_perplexity(audit, p_bigram)
assert math.isclose(corrected_score, corpus_perplexity(audit, p_bigram), rel_tol=1e-12)
print('Independent repair checks passed.')


Independent repair checks passed.


**Your diagnosis:** Replace this text with two or three sentences explaining the bug, the test that exposed it, and the correction. This explanation supports the class discussion; CP6 is the numerical Moodle answer.


### Moodle checkpoint CP6

On audit.txt, using the alpha = 0.1 bigram model, compute the absolute difference between the candidate evaluator's arithmetic mean of sentence perplexities and the correct token-weighted corpus perplexity. Submit abs(candidate_perplexity - corrected_perplexity), rounded to six digits after the decimal point. Include </s> targets as elsewhere in the notebook.


In [38]:
cp6 = abs(candidate_score - corrected_score)
print(f'CP6: {cp6:.6f}')


CP6: 23.806854


## Submit your six checkpoints

Run the following cell after completing the notebook. Copy each result into its matching Moodle question. Keep your notebook as a record of how you obtained and checked the answers.


In [39]:
submission = {'CP1': cp1, 'CP2': cp2, 'CP3': cp3, 'CP4': cp4, 'CP5': cp5, 'CP6': cp6}
for name, value in submission.items():
    if name in {'CP1', 'CP3'}:
        print(f'{name}: {value:d}')
    elif name == 'CP5':
        print(f'{name}: {value:.1f}')
    else:
        print(f'{name}: {value:.6f}')


CP1: 312
CP2: 0.533569
CP3: 395
CP4: 9.359853
CP5: 0.9
CP6: 23.806854


## Optional exploration: greedy generation

No additional checkpoint. Run the supplied decoder below after finishing the main tasks. It chooses the highest-probability next token, with alphabetical tie-breaking. Compare two starting words and consider why a locally likely continuation may be repetitive or uninformative. A maximum length prevents an endless loop.


In [40]:
def greedy_generate(first_word='the', max_new_tokens=20):
    tokens = [first_word if first_word in vocab_set else UNK]
    for _ in range(max_new_tokens):
        history = tokens[-1]
        next_word = min(vocab, key=lambda w: (-p_bigram(history, w), w))
        if next_word == EOS:
            break
        tokens.append(next_word)
    return ' '.join(tokens)

print(greedy_generate('the'))
print(greedy_generate('we'))


the teacher .
we discuss the teacher .


## Closing discussion

- Which checks exposed mistakes that a plausible final number would have hidden?

The perplexity tests caught averaging sentence perplexities instead of pooling the log probabilities and target counts across the corpus—a method that can return a plausible positive number but the wrong metric.

- What information does a bigram model discard?

It conditions each prediction only on the immediately preceding word. It therefore loses longer-range context, such as earlier words that affect meaning or grammar

- Why should preprocessing and evaluation conventions accompany a reported perplexity?

Perplexity depends on what counts as a word and a prediction. The preprocesssing choices such as lowercasing, rare-word handling, the vocabulary, sentence-boundary tokens, and whether </s> is counted can affect the result of the perplexity then. 

**Further reading:** course Lecture 1; Jurafsky and Martin, [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/), material on n-gram language models. Data and exercises here are original teaching material prepared for this course.
